<a href="https://colab.research.google.com/github/Vysokodelovoi/IAD_GP/blob/main/eda_univariate_cat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Univariate analysis категориальных переменных

In [1]:
!pip install country_converter --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 4.4 MB/s eta 0:00:00


In [2]:
!pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 66.3 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
import plotly.graph_objects as go
import country_converter as coco
import pycountry
import gettext

In [4]:
df = pd.read_parquet('main_df.parquet')
df_touragency_full = pd.read_parquet('df_touragency_full.parquet')
df_corporate_tours = pd.read_parquet('df_corporate_tours.parquet')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118898 entries, 0 to 118897
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           118898 non-null  category      
 1   is_canceled                     118898 non-null  int64         
 2   lead_time                       118898 non-null  int64         
 3   arrival_date_year               118898 non-null  int64         
 4   arrival_date_month              118898 non-null  category      
 5   arrival_date_week_number        118898 non-null  int64         
 6   arrival_date_day_of_month       118898 non-null  int64         
 7   stays_in_weekend_nights         118898 non-null  int64         
 8   stays_in_week_nights            118898 non-null  int64         
 9   adults                          118898 non-null  int64         
 10  children                        118898 non-null  int64  

In [6]:
custom_style = go.layout.Template(pio.templates["plotly_white"])

custom_style.layout.update(
    paper_bgcolor="#fcfbfa",
    plot_bgcolor="#f7f5f0",
    font=dict(
        family="Segoe UI, Arial, sans-serif",
        size=13,
        color="#3d4047"
    ),

    margin=dict(t=80, b=60, l=60, r=40),
    title=dict(
        font=dict(size=22, color="#1c212d", weight="bold"),
        pad=dict(b=10)
    )
)

custom_style.layout.colorway = [
    "#deb887",
    "#e28743",
    "#6b8e6b",
    "#e06d75",
    "#205072"
]

axis_config = dict(
    gridcolor="#eae6df",
    zerolinecolor="#d3cbd0",
    tickcolor="#c8becc",
    ticklen=6,
    linecolor="#eae6df",
    showgrid=True,
    ticks="outside",
    title=dict(font=dict(weight="bold", color="#1c212d")),
    tickfont=dict(
        size=12,
        color="#545861",
        family="Arial, sans-serif"
    )
)

custom_style.layout.xaxis = go.layout.XAxis(axis_config)
custom_style.layout.yaxis = go.layout.YAxis(axis_config)

custom_style.layout.legend = dict(
    bgcolor="rgba(252, 251, 250, 0.9)",
    bordercolor="#eae6df",
    borderwidth=1,
    orientation="h",
    yanchor="bottom", y=1.02,
    xanchor="right", x=1
)

custom_style.data.pie = [
    go.Pie(
        textposition="outside",
        textinfo="percent+label",
        textfont=dict(
            size=14,
            color="#2a2c30",
            family="Arial, sans-serif"
        )
    )
]

custom_style.layout.geo = go.layout.Geo(
    showframe=False,
    showcoastlines=True,
    coastlinecolor="#eae6df",
    projection_type="equirectangular",
    landcolor="#f7f5f0",
    lakecolor="#fcfbfa",
    bgcolor="#fcfbfa"
)

pio.templates["summer_minimal"] = custom_style
pio.templates.default = "summer_minimal"

## `hotel`

In [7]:
fig = px.histogram(df,
             x='hotel',
             title='Типы отелей в датасете')

fig.update_layout(
    xaxis_title='тип отеля',
    yaxis_title='количество'
)
fig.show()

## `meal`

In [8]:
meals = df.groupby('meal')['hotel'].count().reset_index()

/tmp/ipykernel_25418/158431835.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [9]:
meals

,meal,hotel
0,BB,91863
1,FB,798
2,HB,14434
3,SC,10638
4,Undefined,1165


В столбце есть значения 'undefined'. Избавимся от них, обновив категории.

In [10]:
df = df[df.meal != 'Undefined'].copy()

In [11]:
df['meal'] = df['meal'].astype('category')

In [12]:
df['meal'] = df['meal'].cat.remove_unused_categories()

In [13]:
meals = df.groupby('meal')['hotel'].count().reset_index()

/tmp/ipykernel_25418/158431835.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [14]:
meals

,meal,hotel
0,BB,91863
1,FB,798
2,HB,14434
3,SC,10638


In [15]:
fig = px.pie(meals,
             values='hotel',
             names='meal',
             title='Типы пансионов',
             labels={
                'BB': 'Завтрак',
                'HB': 'Полупансион',
                'FB': 'Полный пансион',
                'SC': 'Без питания'
            })

leg = {
    'BB': 'Завтрак',
    'HB': 'Полупансион',
    'FB': 'Полный пансион',
    'SC': 'Без питания'
}

fig.for_each_trace(lambda t: t.update(labels=[leg.get(x, x) for x in t.labels]))

fig.show()

## `country`

In [16]:
df.country.unique()

['PRT', 'GBR', 'USA', 'ESP', 'IRL', ..., 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 177
Categories (177, object): ['ABW', 'AGO', 'AIA', 'ALB', ..., 'VNM', 'ZAF', 'ZMB', 'ZWE']

Преобразуем коды стран в полное название, а также переведем названия стран на русский для более удобного восприятия.

In [17]:
import pycountry
import gettext

In [18]:
def get_country_name(code):
    try:
        country = pycountry.countries.get(alpha_3=code)
        return country.name
    except:
        return code

In [19]:
df['country_full'] = df['country'].astype(str).apply(get_country_name)
df['country_full'] = df['country_full'].astype('category')

In [20]:
countries = df.groupby('country_full')['country'].count().reset_index()

/tmp/ipykernel_25418/1566493600.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [21]:
countries

,country_full,country
0,Albania,12
1,Algeria,103
2,American Samoa,1
3,Andorra,7
4,Angola,362
...,...,...
172,"Venezuela, Bolivarian Republic of",26
173,Viet Nam,8
174,"Virgin Islands, British",1
175,Zambia,2


In [22]:
countries = countries.sort_values(by='country', ascending=False)

In [23]:
countries

,country_full,country
130,Portugal,47887
167,United Kingdom,12109
55,France,10324
149,Spain,8271
60,Germany,7286
...,...,...
116,New Caledonia,1
118,Nicaragua,1
151,Sudan,1
169,United States Minor Outlying Islands,1


In [24]:
countries['log_count'] = np.log(countries['country'])

In [25]:
fig = px.choropleth(
    countries,
    locations="country_full",
    locationmode="country names",
    color="log_count",
    color_continuous_scale = "Tealrose",
    title="География гостей в отелях (логнормированная)",
    hover_data={
        "log_count": False,
        "country_full": True,
        "country": True
    },
    labels={
        "country_full": "Страна",
        "country": "Количество гостей",
        "log_count": "логарифм количества гостей"
    }
)

fig.show()

## `arrival_date_month`

Чтобы правильно отсортировать данные на графике, добавлю столбец с номером месяца.

In [26]:
mlist = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
df['arrival_date_month_num'] = df['arrival_date_month'].apply(lambda x: mlist.index(x))

In [27]:
mon = df.groupby(['arrival_date_month_num', 'arrival_date_month'], observed=True)['hotel'].count().reset_index()

In [28]:
mon['arrival_date_month_num'] = mon['arrival_date_month_num'].astype(int)
mon = mon.sort_values('arrival_date_month_num')

In [29]:
mon

,arrival_date_month_num,arrival_date_month,hotel
4,0,January,5746
3,1,February,7781
7,2,March,9566
0,3,April,10886
8,4,May,11767
6,5,June,10878
5,6,July,12585
1,7,August,13809
11,8,September,10463
10,9,October,11093


In [30]:
fig = px.histogram(
    mon,
    x='arrival_date_month',
    y='hotel',
    title='Популярность бронирований по месяцам')

fig.update_layout(
    xaxis_title='месяц',
    yaxis_title='количество бронирований'
)
fig.show()

In [31]:
fig = px.line(
    mon,
    x='arrival_date_month',
    y='hotel',
    title='Популярность бронирований по месяцам',
    category_orders={'arrival_date_month': mon['arrival_date_month'].tolist()}
)

fig.update_traces(line=dict(shape='spline', width=4, color='#205072'))

fig.update_layout(
    xaxis_title='Месяц',
    yaxis_title='Количество бронирований'
)
fig.show()

## `market_segment`

In [32]:
df.market_segment.unique()

['Direct', 'Corporate', 'Online TA', 'Offline TA/TO', 'Complementary', 'Groups', 'Aviation']
Categories (7, object): ['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups',
                         'Offline TA/TO', 'Online TA']

In [33]:
ms = df.groupby('market_segment')['hotel'].count().reset_index()

/tmp/ipykernel_25418/842779170.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [34]:
ms

,market_segment,hotel
0,Aviation,237
1,Complementary,728
2,Corporate,5096
3,Direct,12377
4,Groups,19007
5,Offline TA/TO,23902
6,Online TA,56386


Если график снизу неправильно отображается, уменьшите масштаб страницы.

In [46]:
leg = {
    'Aviation': 'для авиасотрудников',
    'Complementary': 'бесплатная',
    'Corporate': 'командировочная',
    'Direct': 'через отель напрямую',
    'Groups': 'групповое (много номеров)',
    'Offline TA/TO': 'офлайн через туроператора',
    'Online TA': 'онлайн через туроператора'
}
x = leg.values()

fig = px.bar(
    ms,
    y='hotel',
    x = x,
    title = 'Популярность видов бронирований')

fig.update_layout(
    xaxis_title='как забронирован',
    yaxis_title='количество бронирований'
)

fig.show()

## `distribution_channel`

In [48]:
df.distribution_channel.unique()

['Direct', 'Corporate', 'TA/TO', 'Undefined', 'GDS']
Categories (5, object): ['Corporate', 'Direct', 'GDS', 'TA/TO', 'Undefined']

In [50]:
df = df[df.distribution_channel != 'Undefined'].copy()
df['distribution_channel'] = df['distribution_channel'].astype('category')
df['distribution_channel'] = df['distribution_channel'].cat.remove_unused_categories()

In [52]:
df.distribution_channel.unique()

['Direct', 'Corporate', 'TA/TO', 'GDS']
Categories (4, object): ['Corporate', 'Direct', 'GDS', 'TA/TO']

In [54]:
distch = df.groupby('distribution_channel')['hotel'].count().reset_index()

/tmp/ipykernel_25418/4122291981.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [55]:
distch

,distribution_channel,hotel
0,Corporate,6444
1,Direct,14214
2,GDS,193
3,TA/TO,96881


In [62]:
leg = {
    'Corporate': 'командировочная',
    'Direct': 'через отель напрямую',
    'GDS': 'на B2B-портале',
    'TA/TO': 'через туроператора'
}

x = leg.values()

fig = px.pie(
    distch,
    names=x,
    values= distch.hotel,
    title = 'Популярность видов бронирований')

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

In [64]:
df.reserved_room_type.unique()

['C', 'A', 'D', 'E', 'G', 'F', 'H', 'L', 'B', 'P']
Categories (10, object): ['A', 'B', 'C', 'D', ..., 'G', 'H', 'L', 'P']

In [66]:
rt = df.groupby('reserved_room_type')['hotel'].count().reset_index()
rt = rt.sort_values('reserved_room_type')

/tmp/ipykernel_25418/949560664.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [67]:
rt

,reserved_room_type,hotel
0,A,84595
1,B,1114
2,C,907
3,D,19102
4,E,6453
5,F,2879
6,G,2073
7,H,601
8,L,6
9,P,2


In [68]:
rtv = {
    'A': 'стандартный',
    'B': 'улучшенный',
    'C': 'комфорт/делюкс',
    'D': 'семейный',
    'E': 'бизнес/экзекьютив',
    'F': 'полулюкс',
    'G': 'люкс',
    'H': 'президентский люкс',
    'L': 'апартаменты',
    'P': 'промо/эконом'
}
fig = px.pie(
    rt,
    names=rtv,
    values= rt.hotel,
    title = 'Популярность брони видов номеров')

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

## `assigned_room_type`

In [70]:
df.assigned_room_type.unique()

['C', 'A', 'D', 'E', 'G', ..., 'B', 'H', 'L', 'K', 'P']
Length: 12
Categories (12, object): ['A', 'B', 'C', 'D', ..., 'I', 'K', 'L', 'P']

In [71]:
at = df.groupby('assigned_room_type')['hotel'].count().reset_index()
at = at.sort_values('assigned_room_type')

/tmp/ipykernel_25418/1427066157.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [72]:
at

,assigned_room_type,hotel
0,A,73039
1,B,2156
2,C,2251
3,D,25043
4,E,7665
5,F,3707
6,G,2528
7,H,706
8,I,355
9,K,279


In [84]:
atv = {
    'A': 'стандартный',
    'B': 'улучшенный',
    'C': 'комфорт/делюкс',
    'D': 'семейный',
    'E': 'бизнес/экзекьютив',
    'F': 'полулюкс',
    'G': 'люкс',
    'H': 'президентский люкс',
    'I': 'смежный',
    'K': 'с кухней',
    'L': 'апартаменты',
    'P': 'промо/эконом'
}

fig = px.pie(
    at,
    names=atv,
    values= at.hotel,
    title = 'Популярность назначенных видов номеров',
    width=1500,
    height=950)


fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

## `deposit_type`

In [87]:
df.deposit_type.unique()

['No Deposit', 'Refundable', 'Non Refund']
Categories (3, object): ['No Deposit', 'Non Refund', 'Refundable']

In [88]:
dt = df.groupby('deposit_type')['hotel'].count().reset_index()
dt = dt.sort_values('deposit_type')

/tmp/ipykernel_25418/3853387325.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [89]:
dt

,deposit_type,hotel
0,No Deposit,103145
1,Non Refund,14425
2,Refundable,162


In [90]:
dtv = {
    'No Deposit': 'без предоплаты',
    'Non Refund': 'невозвратный',
    'Refundable': 'возвратный',
}

fig = px.pie(
    dt,
    names=dtv,
    values= dt.hotel,
    title = 'Популярность типов предоплаты'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

## `customer_type`

In [92]:
df.customer_type.unique()

['Transient', 'Contract', 'Transient-Party', 'Group']
Categories (4, object): ['Contract', 'Group', 'Transient', 'Transient-Party']

In [94]:
ct = df.groupby('customer_type')['hotel'].count().reset_index()
ct = ct.sort_values('customer_type')

/tmp/ipykernel_25418/3760855327.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [95]:
ct

,customer_type,hotel
0,Contract,4061
1,Group,568
2,Transient,88789
3,Transient-Party,24314


In [96]:
ctv = {
    'Contract': 'договорные/контрактные',
    'Group': 'групповые',
    'Transient': 'индивидуальные (независимые)',
    'Transient-Party': 'связанные индивидуальные'
}

fig = px.pie(
    ct,
    names=ctv,
    values= ct.hotel,
    title = 'Распределение видов клиентов'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

## `reservation_status`

In [98]:
df.reservation_status.unique()

['Check-Out', 'Canceled', 'No-Show']
Categories (3, object): ['Canceled', 'Check-Out', 'No-Show']

In [99]:
rt = df.groupby('reservation_status')['hotel'].count().reset_index()
rt = rt.sort_values('reservation_status')

/tmp/ipykernel_25418/2635271030.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [100]:
rt

,reservation_status,hotel
0,Canceled,42666
1,Check-Out,73865
2,No-Show,1201


In [101]:
rtv = {
    'Canceled': 'отменено',
    'Check-Out': 'успешный выезд',
    'No-Show	': 'неявка',
}

fig = px.pie(
    rt,
    names=rtv,
    values= rt.hotel,
    title = 'Распределение итоговых статусов бронирования'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

# Сохранение измененного файла

In [103]:
df.to_parquet('main_df_edited.parquet')

In [ ]:
#если возникают проблемы
from google.colab import drive
drive.mount('/content/drive')

df.to_parquet('/content/drive/MyDrive/gp_iad/main_df_edited.parquet', index=False)